# Experiment: Resume LaTeX IR Spike

Objective:
- Replace direct LLM LaTeX rewriting with a deterministic parse -> validate -> render -> parse loop for the existing `moderncv` resume template.
- Prove that semantic structure can be extracted into a canonical IR and rendered back into valid LaTeX without losing unsupported content.

Success criteria:
- The current resume parses into predictable `highlights`, `skills`, `experience`, `education`, `projects`, `publications`, and `custom` shapes.
- A rendered document parses back into the same high-level structure.
- Malformed or unknown inputs degrade into warnings and raw/custom preservation instead of silent data loss.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from hashlib import sha1
from pathlib import Path
import json
import pprint
import re
from typing import Any, Iterable, Optional


def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "templates" / "resume_templates" / "data_science_resume.tex").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SAMPLE_RESUME_PATH = REPO_ROOT / "templates" / "resume_templates" / "data_science_resume.tex"
assert SAMPLE_RESUME_PATH.exists(), SAMPLE_RESUME_PATH
SAMPLE_RESUME_PATH


WindowsPath('e:/Workspace/Agentic Job Assistant/ai-job-hunter/templates/resume_templates/data_science_resume.tex')

## Plan

- Hypothesis:
  A resume-specific parser can safely model the current LaTeX template without pretending to support general LaTeX.
- Parsing strategy:
  Split the document, extract header metadata, segment active sections, parse only known resume patterns, and preserve unknowns as raw/custom.
- Validation metrics:
  Section coverage, warning surface area, and round-trip structural stability.


In [2]:
class EscapingMode(str, Enum):
    TRUSTED_LATEX = "trusted_latex"
    PLAIN_TEXT = "plain_text"


class Provenance(str, Enum):
    ACTIVE = "active"
    RAW_FALLBACK = "raw_fallback"
    GENERATED = "generated"


class RenderMode(str, Enum):
    SOURCE_PRESERVED = "source_preserved"
    REGENERATED = "regenerated"


class SectionKind(str, Enum):
    HIGHLIGHTS = "highlights"
    SKILLS = "skills"
    EXPERIENCE = "experience"
    PROJECTS = "projects"
    EDUCATION = "education"
    PUBLICATIONS = "publications"
    CUSTOM = "custom"


SECTION_ALIASES = {
    "highlights": SectionKind.HIGHLIGHTS,
    "summary": SectionKind.HIGHLIGHTS,
    "profile": SectionKind.HIGHLIGHTS,
    "skills": SectionKind.SKILLS,
    "technical skills": SectionKind.SKILLS,
    "experience": SectionKind.EXPERIENCE,
    "professional experience": SectionKind.EXPERIENCE,
    "work experience": SectionKind.EXPERIENCE,
    "projects": SectionKind.PROJECTS,
    "education": SectionKind.EDUCATION,
    "publications": SectionKind.PUBLICATIONS,
}


@dataclass(slots=True)
class TextField:
    value: str
    mode: EscapingMode = EscapingMode.TRUSTED_LATEX


@dataclass(slots=True)
class BulletIR:
    id: str
    text: TextField
    raw_text: str


@dataclass(slots=True)
class EntryIR:
    id: str
    title: Optional[TextField]
    organization: Optional[TextField]
    location: Optional[TextField]
    date_range: Optional[TextField]
    bullets: list[BulletIR] = field(default_factory=list)
    description: Optional[TextField] = None
    hyperlink: Optional[str] = None
    raw_source: str = ""
    render_mode: RenderMode = RenderMode.SOURCE_PRESERVED
    warnings: list[str] = field(default_factory=list)


@dataclass(slots=True)
class SkillsCategoryIR:
    id: str
    name: TextField
    items: list[TextField] = field(default_factory=list)
    raw_source: str = ""


@dataclass(slots=True)
class SectionIR:
    id: str
    kind: SectionKind
    title: str
    order: int
    provenance: Provenance
    raw_source: str
    description: Optional[TextField] = None
    bullets: list[BulletIR] = field(default_factory=list)
    entries: list[EntryIR] = field(default_factory=list)
    skills_categories: list[SkillsCategoryIR] = field(default_factory=list)
    render_mode: RenderMode = RenderMode.SOURCE_PRESERVED
    warnings: list[str] = field(default_factory=list)


@dataclass(slots=True)
class HeaderIR:
    name: Optional[str]
    emails: list[str] = field(default_factory=list)
    phone_numbers: list[str] = field(default_factory=list)
    location: Optional[str] = None
    links: list[dict[str, str]] = field(default_factory=list)
    raw_header_block: str = ""


@dataclass(slots=True)
class DocumentMetaIR:
    source_path: str
    preamble: str
    postamble: str
    parser_warnings: list[str] = field(default_factory=list)
    original_text: str = ""
    original_section_ids: list[str] = field(default_factory=list)


@dataclass(slots=True)
class ResumeIR:
    document_meta: DocumentMetaIR
    header: HeaderIR
    sections: list[SectionIR]


def slugify(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return slug or "item"


def short_hash(value: str) -> str:
    return sha1(value.encode("utf-8")).hexdigest()[:8]


def make_text(value: Optional[str], mode: EscapingMode = EscapingMode.TRUSTED_LATEX) -> Optional[TextField]:
    if value is None:
        return None
    if not value.strip():
        return None
    return TextField(value=value.strip(), mode=mode)


def make_id(prefix: str, *parts: str) -> str:
    joined = "::".join(part for part in parts if part)
    return f"{slugify(prefix)}-{short_hash(joined or prefix)}"


def normalize_section_kind(title: str) -> SectionKind:
    normalized = re.sub(r"\s+", " ", title).strip().lower()
    return SECTION_ALIASES.get(normalized, SectionKind.CUSTOM)


def unwrap_outer_group(value: str) -> str:
    text = value.strip()
    if len(text) >= 2 and text.startswith("{") and text.endswith("}"):
        depth = 0
        for index, char in enumerate(text):
            if char == "{" and (index == 0 or text[index - 1] != "\\"):
                depth += 1
            elif char == "}" and text[index - 1] != "\\":
                depth -= 1
            if depth == 0 and index != len(text) - 1:
                return text
        return text[1:-1].strip()
    return text


In [3]:
def skip_whitespace(text: str, index: int) -> int:
    while index < len(text) and text[index].isspace():
        index += 1
    return index


def read_balanced(text: str, index: int, open_char: str = "{", close_char: str = "}") -> tuple[str, int]:
    if index >= len(text) or text[index] != open_char:
        raise ValueError(f"Expected '{open_char}' at position {index}.")
    depth = 0
    start = index + 1
    cursor = index
    while cursor < len(text):
        char = text[cursor]
        if char == "\\":
            cursor += 2
            continue
        if char == open_char:
            depth += 1
        elif char == close_char:
            depth -= 1
            if depth == 0:
                return text[start:cursor], cursor + 1
        cursor += 1
    raise ValueError(f"Unbalanced {open_char}{close_char} block starting at position {index}.")


def parse_macro_invocation(text: str, start: int, macro_name: str) -> dict[str, Any]:
    token = f"\\{macro_name}"
    if not text.startswith(token, start):
        raise ValueError(f"Expected macro {token!r} at position {start}.")
    cursor = start + len(token)
    cursor = skip_whitespace(text, cursor)
    optional_arg = None
    if cursor < len(text) and text[cursor] == "[":
        optional_arg, cursor = read_balanced(text, cursor, "[", "]")
    args: list[str] = []
    while True:
        cursor = skip_whitespace(text, cursor)
        if cursor < len(text) and text[cursor] == "{":
            arg, cursor = read_balanced(text, cursor)
            args.append(arg)
            continue
        break
    return {
        "macro_name": macro_name,
        "optional_arg": optional_arg,
        "args": args,
        "end": cursor,
        "raw": text[start:cursor],
    }


def remove_visual_noise(text: str) -> str:
    cleaned_lines: list[str] = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if re.fullmatch(r"\\vspace\*?\{[^{}]*\}", line):
            continue
        if re.fullmatch(r"\\setlength\{[^{}]*\}\{[^{}]*\}", line):
            continue
        cleaned_lines.append(raw_line.rstrip())
    cleaned = "\n".join(cleaned_lines).strip()
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    return cleaned.strip()


def split_top_level(text: str, delimiter: str) -> list[str]:
    parts: list[str] = []
    current: list[str] = []
    brace_depth = 0
    bracket_depth = 0
    previous = ""
    for char in text:
        if char == "{" and previous != "\\":
            brace_depth += 1
        elif char == "}" and previous != "\\":
            brace_depth = max(0, brace_depth - 1)
        elif char == "[" and previous != "\\":
            bracket_depth += 1
        elif char == "]" and previous != "\\":
            bracket_depth = max(0, bracket_depth - 1)
        if char == delimiter and brace_depth == 0 and bracket_depth == 0:
            parts.append("".join(current))
            current = []
            previous = char
            continue
        current.append(char)
        previous = char
    parts.append("".join(current))
    return parts


def split_unescaped_ampersand(row: str) -> tuple[str, str]:
    brace_depth = 0
    for index, char in enumerate(row):
        if char == "{" and (index == 0 or row[index - 1] != "\\"):
            brace_depth += 1
        elif char == "}" and row[index - 1] != "\\":
            brace_depth = max(0, brace_depth - 1)
        elif char == "&" and brace_depth == 0 and (index == 0 or row[index - 1] != "\\"):
            return row[:index], row[index + 1 :]
    raise ValueError(f"Could not split skills row: {row!r}")


def extract_href(text: Optional[str]) -> tuple[Optional[str], Optional[str]]:
    if not text:
        return None, None
    stripped = text.strip()
    if not stripped.startswith(r"\href{"):
        return stripped, None
    parsed = parse_macro_invocation(stripped, 0, "href")
    if len(parsed["args"]) < 2:
        return stripped, None
    return unwrap_outer_group(parsed["args"][1]), unwrap_outer_group(parsed["args"][0])


def extract_itemize_blocks(text: str) -> list[tuple[int, int, str]]:
    blocks: list[tuple[int, int, str]] = []
    token = r"\begin{itemize}"
    search_index = 0
    while True:
        start = text.find(token, search_index)
        if start == -1:
            break
        end = text.find(r"\end{itemize}", start)
        if end == -1:
            raise ValueError("Found \\begin{itemize} without a matching \\end{itemize}.")
        end += len(r"\end{itemize}")
        blocks.append((start, end, text[start:end]))
        search_index = end
    return blocks


def parse_itemize_bullets(text: str, provenance: Provenance, prefix: str) -> list[BulletIR]:
    body = text
    if r"\begin{itemize}" in body:
        body = body.replace(r"\begin{itemize}", "", 1)
    if r"\end{itemize}" in body:
        body = body.rsplit(r"\end{itemize}", 1)[0]
    item_pattern = re.compile(r"(?m)^\s*\\item\b")
    matches = list(item_pattern.finditer(body))
    bullets: list[BulletIR] = []
    for index, match in enumerate(matches):
        start = match.end()
        end = matches[index + 1].start() if index + 1 < len(matches) else len(body)
        item_text = remove_visual_noise(body[start:end])
        if not item_text:
            continue
        bullets.append(
            BulletIR(
                id=make_id(prefix, str(index), item_text),
                text=TextField(item_text, EscapingMode.TRUSTED_LATEX),
                raw_text=item_text,
            )
        )
    return bullets


def escape_latex(text: str) -> str:
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in text)


def render_text_field(field: Optional[TextField]) -> str:
    if field is None:
        return ""
    if field.mode == EscapingMode.PLAIN_TEXT:
        return escape_latex(field.value)
    return field.value


def pretty_json(data: Any) -> str:
    return json.dumps(data, indent=2, ensure_ascii=True)


In [4]:
@dataclass(slots=True)
class RawSectionBlock:
    title: str
    kind: SectionKind
    raw_source: str
    content: str
    order: int
    start: int
    end: int


def split_document(text: str) -> tuple[str, str, str]:
    begin_token = r"\begin{document}"
    end_token = r"\end{document}"
    begin = text.find(begin_token)
    end = text.rfind(end_token)
    if begin == -1 or end == -1 or end <= begin:
        raise ValueError("Input does not contain a valid \\begin{document} ... \\end{document} pair.")
    preamble = text[:begin]
    body = text[begin + len(begin_token) : end]
    postamble = text[end + len(end_token) :]
    return preamble, body, postamble


def parse_header(preamble: str, body: str) -> HeaderIR:
    section_match = re.search(r"(?m)^(?!\s*%)\s*\\section\{", body)
    header_block = body[: section_match.start()] if section_match else body

    name_match = re.search(r"\\name\{([^{}]*)\}\{([^{}]*)\}", preamble)
    name = None
    if name_match:
        first = name_match.group(1).strip()
        last = name_match.group(2).strip()
        name = " ".join(part for part in [first, last] if part)

    emails = re.findall(r"mailto:([^}]+)", header_block)
    phone_numbers = sorted({match.strip() for match in re.findall(r"\+?\d[\d()\s.-]{7,}\d", header_block)})

    location_match = re.search(r"\\faHome\\enspace\s*([^\\]+?)\\\\", header_block, re.DOTALL)
    location = location_match.group(1).strip() if location_match else None

    links: list[dict[str, str]] = []
    search_index = 0
    while True:
        href_start = header_block.find(r"\href{", search_index)
        if href_start == -1:
            break
        try:
            href = parse_macro_invocation(header_block, href_start, "href")
        except ValueError:
            search_index = href_start + len(r"\href")
            continue
        if len(href["args"]) >= 2:
            url = unwrap_outer_group(href["args"][0]).strip()
            label = unwrap_outer_group(href["args"][1]).strip()
            label = re.sub(r"\\textcolor\{[^{}]+\}\{([^{}]+)\}", r"\1", label)
            label = unwrap_outer_group(label).strip()
            links.append({"label": label, "url": url})
        search_index = href["end"]

    return HeaderIR(
        name=name,
        emails=emails,
        phone_numbers=phone_numbers,
        location=location,
        links=links,
        raw_header_block=header_block.strip("\n"),
    )


def segment_sections(body: str) -> list[RawSectionBlock]:
    pattern = re.compile(r"(?m)^(?!\s*%)\s*\\section\{")
    matches = list(pattern.finditer(body))
    sections: list[RawSectionBlock] = []
    for order, match in enumerate(matches):
        brace_start = body.find("{", match.start())
        title, title_end = read_balanced(body, brace_start)
        section_start = match.start()
        section_end = matches[order + 1].start() if order + 1 < len(matches) else len(body)
        raw_source = body[section_start:section_end].strip("\n")
        content = body[title_end:section_end].strip("\n")
        sections.append(
            RawSectionBlock(
                title=title.strip(),
                kind=normalize_section_kind(title),
                raw_source=raw_source,
                content=content,
                order=order,
                start=section_start,
                end=section_end,
            )
        )
    return sections


In [5]:
def entry_body_to_fields(body: str, provenance: Provenance) -> tuple[Optional[TextField], Optional[TextField], Optional[TextField], list[BulletIR]]:
    cleaned_body = body
    itemize_blocks = extract_itemize_blocks(cleaned_body)
    description = None
    bullets: list[BulletIR] = []
    prose_chunks: list[str] = []
    if itemize_blocks:
        first_start, first_end, first_block = itemize_blocks[0]
        before = remove_visual_noise(cleaned_body[:first_start])
        after = remove_visual_noise(cleaned_body[first_end:])
        if before:
            prose_chunks.append(before)
        if after:
            prose_chunks.append(after)
        bullets = parse_itemize_bullets(first_block, provenance, prefix="bullet")
    else:
        remaining = remove_visual_noise(cleaned_body)
        if remaining:
            prose_chunks.append(remaining)
    if prose_chunks:
        description = TextField("\n\n".join(prose_chunks), EscapingMode.TRUSTED_LATEX)

    organization = None
    location = None
    if description and r"\hfill" in description.value:
        left, right = description.value.split(r"\hfill", 1)
        organization = make_text(left)
        cleaned_right = re.sub(r"\\textit\{([^{}]+)\}", r"\1", right).strip()
        location = make_text(unwrap_outer_group(cleaned_right))
        description = None
    return organization, location, description, bullets


def parse_entry_section(block: RawSectionBlock, provenance: Provenance) -> SectionIR:
    active_content = block.content
    entry_positions = [match.start() for match in re.finditer(r"\\customcventry", active_content)]
    entries: list[EntryIR] = []
    warnings: list[str] = []

    if not entry_positions:
        return SectionIR(
            id=make_id("section", block.title, str(block.order), provenance.value),
            kind=SectionKind.CUSTOM,
            title=block.title,
            order=block.order,
            provenance=Provenance.RAW_FALLBACK if provenance == Provenance.ACTIVE else provenance,
            raw_source=block.raw_source,
            warnings=[f"No active \\customcventry macros were found in section '{block.title}'."],
        )

    for index, start in enumerate(entry_positions):
        try:
            parsed = parse_macro_invocation(active_content, start, "customcventry")
        except ValueError as exc:
            warnings.append(f"Failed to parse entry {index} in '{block.title}': {exc}")
            continue
        args = parsed["args"]
        if len(args) < 2:
            warnings.append(f"Entry {index} in '{block.title}' did not expose enough arguments.")
            continue

        raw_date = unwrap_outer_group(args[0]) if args else ""
        raw_org = unwrap_outer_group(args[1]) if len(args) >= 2 else ""
        raw_title = unwrap_outer_group(args[2]) if len(args) >= 3 else ""
        raw_location = unwrap_outer_group(args[3]) if len(args) >= 4 else ""
        raw_body = args[-1] if args else ""

        parsed_title, hyperlink = extract_href(raw_title)
        try:
            org_from_body, location_from_body, description, bullets = entry_body_to_fields(raw_body, provenance)
        except ValueError as exc:
            warnings.append(f"Failed to parse entry body {index} in '{block.title}': {exc}")
            org_from_body, location_from_body = None, None
            description = make_text(remove_visual_noise(raw_body))
            bullets = []

        organization = make_text(raw_org) or org_from_body
        location = make_text(raw_location) or location_from_body
        entry = EntryIR(
            id=make_id(block.title, str(index), parsed.get("raw", raw_body)),
            title=make_text(parsed_title),
            organization=organization,
            location=location,
            date_range=make_text(raw_date),
            bullets=[
                BulletIR(
                    id=make_id("bullet", str(index), str(bullet_index), bullet.text.value),
                    text=bullet.text,
                    raw_text=bullet.raw_text,
                )
                for bullet_index, bullet in enumerate(bullets)
            ],
            description=description,
            hyperlink=hyperlink,
            raw_source=parsed["raw"],
        )
        entries.append(entry)

    if warnings and not entries:
        return SectionIR(
            id=make_id("section", block.title, str(block.order), "raw"),
            kind=SectionKind.CUSTOM,
            title=block.title,
            order=block.order,
            provenance=Provenance.RAW_FALLBACK if provenance == Provenance.ACTIVE else provenance,
            raw_source=block.raw_source,
            warnings=warnings,
        )

    return SectionIR(
        id=make_id("section", block.title, str(block.order), provenance.value),
        kind=block.kind,
        title=block.title,
        order=block.order,
        provenance=provenance,
        raw_source=block.raw_source,
        entries=entries,
        warnings=warnings,
    )


def parse_highlights_section(block: RawSectionBlock, provenance: Provenance) -> SectionIR:
    active_content = block.content
    itemize_blocks = extract_itemize_blocks(active_content)
    if not itemize_blocks:
        return SectionIR(
            id=make_id("section", block.title, str(block.order), "raw"),
            kind=SectionKind.CUSTOM,
            title=block.title,
            order=block.order,
            provenance=Provenance.RAW_FALLBACK if provenance == Provenance.ACTIVE else provenance,
            raw_source=block.raw_source,
            warnings=[f"Highlights section '{block.title}' did not contain an itemize block."],
        )
    bullets = parse_itemize_bullets(
        itemize_blocks[0][2],
        provenance,
        prefix=f"{block.title}-bullet",
    )
    return SectionIR(
        id=make_id("section", block.title, str(block.order), provenance.value),
        kind=block.kind,
        title=block.title,
        order=block.order,
        provenance=provenance,
        raw_source=block.raw_source,
        bullets=bullets,
    )


def parse_skills_section(block: RawSectionBlock, provenance: Provenance) -> SectionIR:
    active_content = block.content
    table_match = re.search(r"\\begin\{tabularx\}.*?\\end\{tabularx\}", active_content, re.DOTALL)
    if not table_match:
        return SectionIR(
            id=make_id("section", block.title, str(block.order), "raw"),
            kind=SectionKind.CUSTOM,
            title=block.title,
            order=block.order,
            provenance=Provenance.RAW_FALLBACK if provenance == Provenance.ACTIVE else provenance,
            raw_source=block.raw_source,
            warnings=[f"Skills section '{block.title}' did not contain a tabularx block."],
        )
    table_body = table_match.group(0)
    cursor = table_body.find(r"\begin")
    _env_name, cursor = read_balanced(table_body, cursor + len(r"\begin"))
    cursor = skip_whitespace(table_body, cursor)
    _width_arg, cursor = read_balanced(table_body, cursor)
    cursor = skip_whitespace(table_body, cursor)
    _column_arg, cursor = read_balanced(table_body, cursor)
    inner = table_body[cursor : table_body.rfind(r"\end{tabularx}")]
    rows = [remove_visual_noise(chunk) for chunk in re.split(r"\\\\(?:\[[^\]]+\])?", inner) if remove_visual_noise(chunk)]
    categories: list[SkillsCategoryIR] = []
    warnings: list[str] = []
    for index, row in enumerate(rows):
        try:
            raw_name, raw_values = split_unescaped_ampersand(row)
        except ValueError as exc:
            warnings.append(str(exc))
            continue
        items = [make_text(part) for part in split_top_level(raw_values, ",")]
        categories.append(
            SkillsCategoryIR(
                id=make_id("skills", str(index), raw_name, raw_values),
                name=TextField(unwrap_outer_group(raw_name.strip()), EscapingMode.TRUSTED_LATEX),
                items=[item for item in items if item],
                raw_source=row,
            )
        )
    if warnings and not categories:
        return SectionIR(
            id=make_id("section", block.title, str(block.order), "raw"),
            kind=SectionKind.CUSTOM,
            title=block.title,
            order=block.order,
            provenance=Provenance.RAW_FALLBACK if provenance == Provenance.ACTIVE else provenance,
            raw_source=block.raw_source,
            warnings=warnings,
        )
    return SectionIR(
        id=make_id("section", block.title, str(block.order), provenance.value),
        kind=block.kind,
        title=block.title,
        order=block.order,
        provenance=provenance,
        raw_source=block.raw_source,
        skills_categories=categories,
        warnings=warnings,
    )


In [6]:
def parse_publications_section(block: RawSectionBlock, provenance: Provenance) -> SectionIR:
    active_content = block.content
    itemize_blocks = extract_itemize_blocks(active_content)
    description = None
    bullets: list[BulletIR] = []
    if itemize_blocks:
        first_start, first_end, first_block = itemize_blocks[0]
        header_text = remove_visual_noise(active_content[:first_start])
        tail_text = remove_visual_noise(active_content[first_end:])
        prose_parts = [part for part in [header_text, tail_text] if part]
        description = make_text("\n\n".join(prose_parts))
        bullets = parse_itemize_bullets(
            first_block,
            provenance,
            prefix=f"{block.title}-bullet",
        )
    else:
        description = make_text(remove_visual_noise(active_content))
    return SectionIR(
        id=make_id("section", block.title, str(block.order), provenance.value),
        kind=block.kind,
        title=block.title,
        order=block.order,
        provenance=provenance,
        raw_source=block.raw_source,
        description=description,
        bullets=bullets,
    )


def parse_custom_section(block: RawSectionBlock, provenance: Provenance) -> SectionIR:
    effective_provenance = provenance if provenance != Provenance.ACTIVE else Provenance.RAW_FALLBACK
    return SectionIR(
        id=make_id("section", block.title, str(block.order), effective_provenance.value),
        kind=SectionKind.CUSTOM,
        title=block.title,
        order=block.order,
        provenance=effective_provenance,
        raw_source=block.raw_source,
        description=make_text(remove_visual_noise(block.content)),
    )


def parse_section(block: RawSectionBlock, provenance: Provenance) -> SectionIR:
    if block.kind == SectionKind.HIGHLIGHTS:
        return parse_highlights_section(block, provenance)
    if block.kind == SectionKind.SKILLS:
        return parse_skills_section(block, provenance)
    if block.kind in {SectionKind.EXPERIENCE, SectionKind.EDUCATION, SectionKind.PROJECTS}:
        return parse_entry_section(block, provenance)
    if block.kind == SectionKind.PUBLICATIONS:
        return parse_publications_section(block, provenance)
    return parse_custom_section(block, provenance)


In [7]:
def parse_resume(text: str, source_path: Path) -> ResumeIR:
    preamble, body, postamble = split_document(text)
    header = parse_header(preamble, body)
    raw_sections = segment_sections(body)
    parsed_sections = [parse_section(section, Provenance.ACTIVE) for section in raw_sections]
    parser_warnings = [warning for section in parsed_sections for warning in section.warnings]
    return ResumeIR(
        document_meta=DocumentMetaIR(
            source_path=str(source_path),
            preamble=preamble.rstrip(),
            postamble=postamble.lstrip(),
            parser_warnings=parser_warnings,
            original_text=text,
            original_section_ids=[section.id for section in parsed_sections],
        ),
        header=header,
        sections=parsed_sections,
    )


In [8]:
def validate_resume_ir(resume: ResumeIR) -> None:
    section_ids = [section.id for section in resume.sections]
    if len(section_ids) != len(set(section_ids)):
        raise ValueError("Section ids must be unique.")
    expected_order = list(range(len(resume.sections)))
    actual_order = [section.order for section in resume.sections]
    if actual_order != expected_order:
        raise ValueError(f"Section order drifted: expected {expected_order}, saw {actual_order}.")
    for section in resume.sections:
        if not isinstance(section.kind, SectionKind):
            raise TypeError(f"Invalid section kind for {section.title!r}: {section.kind!r}")
        if not isinstance(section.render_mode, RenderMode):
            raise TypeError(f"Invalid render mode for section {section.title!r}: {section.render_mode!r}")
        for entry in section.entries:
            if not entry.id:
                raise ValueError(f"Entry in section {section.title!r} is missing an id.")
            if not isinstance(entry.render_mode, RenderMode):
                raise TypeError(f"Invalid render mode for entry {entry.id!r}: {entry.render_mode!r}")
            bullet_ids = [bullet.id for bullet in entry.bullets]
            if len(bullet_ids) != len(set(bullet_ids)):
                raise ValueError(f"Duplicate bullet ids found inside entry {entry.id!r}.")
            for bullet in entry.bullets:
                if not isinstance(bullet.text, TextField):
                    raise TypeError(f"Bullet {bullet.id!r} is missing a TextField payload.")
        for bullet in section.bullets:
            if not isinstance(bullet.text, TextField):
                raise TypeError(f"Bullet {bullet.id!r} in section {section.title!r} is invalid.")
        for category in section.skills_categories:
            if not category.items:
                raise ValueError(f"Skills category {category.id!r} has no items.")


def render_bullet_list(bullets: list[BulletIR], spacing: str = "-4pt") -> str:
    lines = [r"\begin{itemize}[leftmargin=0.6cm, label={\textbullet}]"]
    for bullet in bullets:
        lines.append(f"    \\item {render_text_field(bullet.text)}")
        lines.append(f"    \\vspace{{{spacing}}}")
    lines.append(r"\end{itemize}")
    return "\n".join(lines)


def render_entry_title(entry: EntryIR) -> str:
    title = render_text_field(entry.title)
    if entry.hyperlink:
        return f"\\href{{{entry.hyperlink}}}{{{title}}}"
    return title


def render_regenerated_entry(entry: EntryIR, section_kind: SectionKind) -> str:
    date_tex = render_text_field(entry.date_range)
    title_tex = render_entry_title(entry)
    organization_tex = render_text_field(entry.organization)
    location_tex = render_text_field(entry.location)

    if section_kind == SectionKind.EXPERIENCE:
        body_lines: list[str] = [r"\vspace{4pt}"]
        if organization_tex and location_tex:
            body_lines.append(f"{organization_tex} \\hfill \\textit{{{location_tex}}}")
        elif organization_tex:
            body_lines.append(organization_tex)
        elif location_tex:
            body_lines.append(f"\\textit{{{location_tex}}}")
        if entry.description or entry.bullets:
            body_lines.append(r"\vspace{2pt}")
        if entry.description:
            body_lines.append(render_text_field(entry.description))
        if entry.bullets:
            body_lines.append(render_bullet_list(entry.bullets, spacing="-4pt"))
        body = "\n".join(line for line in body_lines if line).strip()
        return (
            "\\customcventry"
            f"{{{date_tex}}}{{}}{{{title_tex}}}{{}}{{}}"
            + "{\n" + body + "\n}"
        )

    if section_kind == SectionKind.EDUCATION:
        body_lines: list[str] = []
        if entry.description or entry.bullets:
            body_lines.append(r"\vspace{2pt}")
        if entry.description:
            body_lines.append(render_text_field(entry.description))
        if entry.bullets:
            body_lines.append(render_bullet_list(entry.bullets, spacing="-4pt"))
        body = "\n".join(line for line in body_lines if line).strip()
        return (
            "\\customcventry"
            f"{{{date_tex}}}{{{{{organization_tex}}}}}{{{title_tex}}}{{{location_tex}}}{{}}{{}}"
            + "{\n" + body + "\n}"
        )

    if section_kind == SectionKind.PROJECTS:
        body_lines: list[str] = []
        if entry.description or entry.bullets:
            body_lines.append(r"\vspace{2pt}")
        if entry.description:
            body_lines.append(render_text_field(entry.description))
        if entry.bullets:
            body_lines.append(render_bullet_list(entry.bullets, spacing="-2pt"))
        body = "\n".join(line for line in body_lines if line).strip()
        return (
            "\\customcventry"
            f"{{{date_tex}}}{{}}{{{title_tex}}}{{}}{{}}"
            + "{\n" + body + "\n}"
        )

    body_parts: list[str] = []
    if entry.description:
        body_parts.append(r"\vspace{2pt}")
        body_parts.append(render_text_field(entry.description))
    if entry.bullets:
        if not body_parts:
            body_parts.append(r"\vspace{2pt}")
        body_parts.append(render_bullet_list(entry.bullets, spacing="-4pt"))
    body = "\n".join(part for part in body_parts if part).strip()
    return (
        "\\customcventry"
        f"{{{date_tex}}}"
        f"{{{organization_tex}}}"
        f"{{{title_tex}}}"
        f"{{{location_tex}}}"
        "{}"
        f"{{{body}}}"
    )


def render_section(section: SectionIR) -> str:
    if section.render_mode == RenderMode.SOURCE_PRESERVED and section.raw_source.strip():
        return section.raw_source.strip("\n")
    if section.kind == SectionKind.CUSTOM:
        return section.raw_source.strip()
    lines = [f"\\section{{{section.title}}}"]
    if section.kind == SectionKind.HIGHLIGHTS:
        lines.extend(
            [
                r"\setlength{\itemsep}{0pt}",
                r"\setlength{\parskip}{0pt}",
                render_bullet_list(section.bullets, spacing="-4pt"),
            ]
        )
    elif section.kind == SectionKind.SKILLS:
        lines.extend(
            [
                r"\vspace{-4pt}",
                r"\begin{tabularx}{\textwidth}{>{\raggedright\arraybackslash\bfseries}p{5.5cm}@{\hspace{0cm}}X}",
            ]
        )
        for category in section.skills_categories:
            rendered_items = ", ".join(render_text_field(item) for item in category.items)
            lines.append(f"{render_text_field(category.name)} & {rendered_items} \\\\[4pt]")
        lines.append(r"\end{tabularx}")
        lines.append(r"\vspace{-16pt}")
    elif section.kind in {SectionKind.EXPERIENCE, SectionKind.EDUCATION, SectionKind.PROJECTS}:
        spacer = r"\vspace{6pt}" if section.kind != SectionKind.EDUCATION else r"\vspace{10pt}"
        rendered_entries = []
        for entry in section.entries:
            if entry.render_mode == RenderMode.SOURCE_PRESERVED and entry.raw_source.strip():
                rendered_entries.append(entry.raw_source.strip("\n"))
            else:
                rendered_entries.append(render_regenerated_entry(entry, section.kind))
        lines.append("\n\n".join(rendered_entries))
        lines.append(spacer)
    elif section.kind == SectionKind.PUBLICATIONS:
        lines.extend([r"\setlength{\itemsep}{0pt}", r"\setlength{\parskip}{0pt}"])
        if section.description:
            lines.append(render_text_field(section.description))
        if section.bullets:
            lines.append(render_bullet_list(section.bullets, spacing="-4pt"))
    return "\n".join(line for line in lines if line).strip()


def can_render_from_original_source(resume: ResumeIR) -> bool:
    if not resume.document_meta.original_text:
        return False
    if any(section.render_mode != RenderMode.SOURCE_PRESERVED for section in resume.sections):
        return False
    current_section_ids = [section.id for section in resume.sections]
    return current_section_ids == resume.document_meta.original_section_ids


def render_resume(resume: ResumeIR) -> str:
    if can_render_from_original_source(resume):
        return resume.document_meta.original_text

    parts = [resume.document_meta.preamble.rstrip(), r"\begin{document}"]
    if resume.header.raw_header_block:
        parts.append(resume.header.raw_header_block.strip())
    for section in resume.sections:
        parts.append(render_section(section))
    parts.append(r"\end{document}")
    if resume.document_meta.postamble.strip():
        parts.append(resume.document_meta.postamble.strip())
    return "\n\n".join(part for part in parts if part).strip() + "\n"


def snapshot_resume(resume: ResumeIR) -> dict[str, Any]:
    return {
        "sections": [
            {
                "kind": section.kind.value,
                "title": section.title,
                "entry_titles": [render_text_field(entry.title) for entry in section.entries],
                "entry_dates": [render_text_field(entry.date_range) for entry in section.entries],
                "bullet_count": len(section.bullets),
                "entry_bullet_counts": [len(entry.bullets) for entry in section.entries],
                "skills_categories": [render_text_field(category.name) for category in section.skills_categories],
                "warnings": section.warnings,
            }
            for section in resume.sections
        ],
        "warning_count": len(resume.document_meta.parser_warnings),
    }


def snapshot_active_structure(resume: ResumeIR) -> dict[str, Any]:
    return {
        "sections": [
            {
                "kind": section.kind.value,
                "title": section.title,
                "entry_titles": [render_text_field(entry.title) for entry in section.entries],
                "entry_dates": [render_text_field(entry.date_range) for entry in section.entries],
                "bullet_count": len(section.bullets),
                "entry_bullet_counts": [len(entry.bullets) for entry in section.entries],
                "skills_categories": [render_text_field(category.name) for category in section.skills_categories],
            }
            for section in resume.sections
        ]
    }


def text_field_to_dict(field_value: Optional[TextField]) -> Optional[dict[str, Any]]:
    if field_value is None:
        return None
    return {
        "value": field_value.value,
        "mode": field_value.mode.value,
    }


def text_field_from_dict(data: Optional[dict[str, Any]]) -> Optional[TextField]:
    if data is None:
        return None
    return TextField(
        value=data["value"],
        mode=EscapingMode(data["mode"]),
    )


def bullet_to_dict(bullet: BulletIR) -> dict[str, Any]:
    return {
        "id": bullet.id,
        "text": text_field_to_dict(bullet.text),
        "raw_text": bullet.raw_text,
    }


def bullet_from_dict(data: dict[str, Any]) -> BulletIR:
    return BulletIR(
        id=data["id"],
        text=text_field_from_dict(data["text"]),
        raw_text=data["raw_text"],
    )


def entry_to_dict(entry: EntryIR) -> dict[str, Any]:
    return {
        "id": entry.id,
        "title": text_field_to_dict(entry.title),
        "organization": text_field_to_dict(entry.organization),
        "location": text_field_to_dict(entry.location),
        "date_range": text_field_to_dict(entry.date_range),
        "bullets": [bullet_to_dict(bullet) for bullet in entry.bullets],
        "description": text_field_to_dict(entry.description),
        "hyperlink": entry.hyperlink,
        "raw_source": entry.raw_source,
        "render_mode": entry.render_mode.value,
        "warnings": entry.warnings,
    }


def entry_from_dict(data: dict[str, Any]) -> EntryIR:
    return EntryIR(
        id=data["id"],
        title=text_field_from_dict(data.get("title")),
        organization=text_field_from_dict(data.get("organization")),
        location=text_field_from_dict(data.get("location")),
        date_range=text_field_from_dict(data.get("date_range")),
        bullets=[bullet_from_dict(item) for item in data.get("bullets", [])],
        description=text_field_from_dict(data.get("description")),
        hyperlink=data.get("hyperlink"),
        raw_source=data.get("raw_source", ""),
        render_mode=RenderMode(data.get("render_mode", RenderMode.SOURCE_PRESERVED.value)),
        warnings=list(data.get("warnings", [])),
    )


def skills_category_to_dict(category: SkillsCategoryIR) -> dict[str, Any]:
    return {
        "id": category.id,
        "name": text_field_to_dict(category.name),
        "items": [text_field_to_dict(item) for item in category.items],
        "raw_source": category.raw_source,
    }


def skills_category_from_dict(data: dict[str, Any]) -> SkillsCategoryIR:
    return SkillsCategoryIR(
        id=data["id"],
        name=text_field_from_dict(data["name"]),
        items=[text_field_from_dict(item) for item in data.get("items", [])],
        raw_source=data.get("raw_source", ""),
    )


def section_to_dict(section: SectionIR) -> dict[str, Any]:
    return {
        "id": section.id,
        "kind": section.kind.value,
        "title": section.title,
        "order": section.order,
        "provenance": section.provenance.value,
        "raw_source": section.raw_source,
        "description": text_field_to_dict(section.description),
        "bullets": [bullet_to_dict(bullet) for bullet in section.bullets],
        "entries": [entry_to_dict(entry) for entry in section.entries],
        "skills_categories": [skills_category_to_dict(category) for category in section.skills_categories],
        "render_mode": section.render_mode.value,
        "warnings": section.warnings,
    }


def section_from_dict(data: dict[str, Any]) -> SectionIR:
    return SectionIR(
        id=data["id"],
        kind=SectionKind(data["kind"]),
        title=data["title"],
        order=data["order"],
        provenance=Provenance(data["provenance"]),
        raw_source=data["raw_source"],
        description=text_field_from_dict(data.get("description")),
        bullets=[bullet_from_dict(item) for item in data.get("bullets", [])],
        entries=[entry_from_dict(item) for item in data.get("entries", [])],
        skills_categories=[skills_category_from_dict(item) for item in data.get("skills_categories", [])],
        render_mode=RenderMode(data.get("render_mode", RenderMode.SOURCE_PRESERVED.value)),
        warnings=list(data.get("warnings", [])),
    )


def resume_to_dict(resume: ResumeIR) -> dict[str, Any]:
    return {
        "document_meta": {
            "source_path": resume.document_meta.source_path,
            "preamble": resume.document_meta.preamble,
            "postamble": resume.document_meta.postamble,
            "parser_warnings": resume.document_meta.parser_warnings,
            "original_text": resume.document_meta.original_text,
            "original_section_ids": resume.document_meta.original_section_ids,
        },
        "header": {
            "name": resume.header.name,
            "emails": resume.header.emails,
            "phone_numbers": resume.header.phone_numbers,
            "location": resume.header.location,
            "links": resume.header.links,
            "raw_header_block": resume.header.raw_header_block,
        },
        "sections": [section_to_dict(section) for section in resume.sections],
    }


def resume_from_dict(data: dict[str, Any]) -> ResumeIR:
    document_meta = data["document_meta"]
    header = data["header"]
    return ResumeIR(
        document_meta=DocumentMetaIR(
            source_path=document_meta["source_path"],
            preamble=document_meta["preamble"],
            postamble=document_meta["postamble"],
            parser_warnings=list(document_meta.get("parser_warnings", [])),
            original_text=document_meta.get("original_text", ""),
            original_section_ids=list(document_meta.get("original_section_ids", [])),
        ),
        header=HeaderIR(
            name=header.get("name"),
            emails=list(header.get("emails", [])),
            phone_numbers=list(header.get("phone_numbers", [])),
            location=header.get("location"),
            links=list(header.get("links", [])),
            raw_header_block=header.get("raw_header_block", ""),
        ),
        sections=[section_from_dict(item) for item in data.get("sections", [])],
    )


def write_json(path: Path, payload: Any) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


In [9]:
sample_resume_text = SAMPLE_RESUME_PATH.read_text(encoding="utf-8")
resume_ir = parse_resume(sample_resume_text, SAMPLE_RESUME_PATH)
validate_resume_ir(resume_ir)

summary = {
    "source": resume_ir.document_meta.source_path,
    "header_name": resume_ir.header.name,
    "section_order": [section.title for section in resume_ir.sections],
    "section_kinds": [section.kind.value for section in resume_ir.sections],
    "warning_count": len(resume_ir.document_meta.parser_warnings),
}
pprint.pp(summary)


{'source': 'e:\\Workspace\\Agentic Job '
           'Assistant\\ai-job-hunter\\templates\\resume_templates\\data_science_resume.tex',
 'header_name': 'Koushik Sivarama Krishnan',
 'section_order': ['Highlights',
                   'Skills',
                   'Experience',
                   'Education',
                   'Projects',
                   'Publications'],
 'section_kinds': ['highlights',
                   'skills',
                   'experience',
                   'education',
                   'projects',
                   'publications'],
 'warning_count': 0}


In [ ]:
ARTIFACT_DIR = REPO_ROOT / "tmp" / "resume_ir_roundtrip"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

resume_ir_payload = resume_to_dict(resume_ir)
parsed_snapshot = snapshot_active_structure(resume_ir)
serialized_rendered_resume = render_resume(resume_ir)

write_json(ARTIFACT_DIR / "resume_ir.json", resume_ir_payload)
write_json(ARTIFACT_DIR / "parsed_snapshot.json", parsed_snapshot)
(ARTIFACT_DIR / "rendered_resume.tex").write_text(serialized_rendered_resume, encoding="utf-8")

reloaded_resume_ir = resume_from_dict(json.loads((ARTIFACT_DIR / "resume_ir.json").read_text(encoding="utf-8")))
assert snapshot_active_structure(reloaded_resume_ir) == parsed_snapshot

artifact_summary = {
    "artifact_dir": str(ARTIFACT_DIR),
    "saved_files": [
        "resume_ir.json",
        "parsed_snapshot.json",
        "rendered_resume.tex",
    ],
    "reload_snapshot_match": True,
}
pprint.pp(artifact_summary)


In [14]:
section_inspection = []
for section in resume_ir.sections:
    section_summary = {
        "id": section.id,
        "kind": section.kind.value,
        "title": section.title,
        "order": section.order,
        "warning_count": len(section.warnings),
        "bullets": [bullet.text.value for bullet in section.bullets],
        "entries": [
            {
                "id": entry.id,
                "title": entry.title.value if entry.title else None,
                "organization": entry.organization.value if entry.organization else None,
                "location": entry.location.value if entry.location else None,
                "date_range": entry.date_range.value if entry.date_range else None,
                "description": entry.description.value if entry.description else None,
                "hyperlink": entry.hyperlink,
                "bullets": [bullet.text.value for bullet in entry.bullets],
            }
            for entry in section.entries
        ],
        "skills_categories": [
            {
                "name": category.name.value,
                "items": [item.value for item in category.items],
            }
            for category in section.skills_categories
        ],
    }
    section_inspection.append(section_summary)

print(
    pretty_json(
        {
            "header": {
                "name": resume_ir.header.name,
                "emails": resume_ir.header.emails,
                "phone_numbers": resume_ir.header.phone_numbers,
                "location": resume_ir.header.location,
                "links": resume_ir.header.links,
            },
            "sections": section_inspection,
        }
    )
)


Master’s in Applied Data Science with \textbf{3+ years} of experience using \textbf{Python}, \textbf{SQL}, and statistical modeling to clean complex data, build reliable features, and support production analytics and machine learning workflows.


## Results

- The inspection cell above dumps the active header, sections, bullets, entries, and skill categories so the extracted IR can be checked directly against the source resume.
- The round-trip cell below now checks both structural stability and exact source-preserving LaTeX equality for unchanged input.
- If a later rewrite step marks a section or entry as regenerated, the renderer will fall back to deterministic reconstruction only for those edited fragments.


In [10]:
import difflib

rendered_resume = render_resume(resume_ir)
reparsed_resume = parse_resume(rendered_resume, SAMPLE_RESUME_PATH.with_name("rendered_resume.tex"))
validate_resume_ir(reparsed_resume)

parsed_snapshot = snapshot_active_structure(resume_ir)
reparsed_snapshot = snapshot_active_structure(reparsed_resume)

assert parsed_snapshot == reparsed_snapshot
exact_latex_match = sample_resume_text == rendered_resume
assert exact_latex_match

diff_preview = ""
if not exact_latex_match:
    diff_lines = list(
        difflib.unified_diff(
            sample_resume_text.splitlines(),
            rendered_resume.splitlines(),
            fromfile="source",
            tofile="rendered",
            n=1,
        )
    )
    diff_preview = "\n".join(diff_lines[:120])

print("Round-trip snapshot is structurally stable.")
print(pretty_json(parsed_snapshot))
print({
    "exact_latex_match": exact_latex_match,
    "source_length": len(sample_resume_text),
    "rendered_length": len(rendered_resume),
})
if diff_preview:
    print(diff_preview)


Round-trip snapshot is structurally stable.
{
  "sections": [
    {
      "kind": "highlights",
      "title": "Highlights",
      "entry_titles": [],
      "entry_dates": [],
      "bullet_count": 3,
      "entry_bullet_counts": [],
      "skills_categories": []
    },
    {
      "kind": "skills",
      "title": "Skills",
      "entry_titles": [],
      "entry_dates": [],
      "bullet_count": 0,
      "entry_bullet_counts": [],
      "skills_categories": [
        "Programming",
        "Data Quality",
        "Statistical Analysis \\& ML",
        "Data Visualization",
        "Data Engineering",
        "Collaboration \\& Documentation"
      ]
    },
    {
      "kind": "experience",
      "title": "Experience",
      "entry_titles": [
        "Co-op Data Scientist",
        "Research Engine Developer",
        "Founding Machine Learning Engineer",
        "Computer Vision Intern",
        "Deep Learning Intern"
      ],
      "entry_dates": [
        "09/2024 \u2010 08/2025",
  

In [ ]:
# Inline resilience and regression checks.

assert [section.title for section in resume_ir.sections] == [
    "Highlights",
    "Skills",
    "Experience",
    "Education",
    "Projects",
    "Publications",
]
assert [section.kind for section in resume_ir.sections] == [
    SectionKind.HIGHLIGHTS,
    SectionKind.SKILLS,
    SectionKind.EXPERIENCE,
    SectionKind.EDUCATION,
    SectionKind.PROJECTS,
    SectionKind.PUBLICATIONS,
]
assert render_resume(resume_ir) == sample_resume_text

custom_fixture = r"""
\documentclass{article}
\begin{document}
\section{Awards}
\smallskip
Won something unusual.
\end{document}
"""
custom_ir = parse_resume(custom_fixture, Path("custom_fixture.tex"))
assert custom_ir.sections[0].kind == SectionKind.CUSTOM
assert render_resume(custom_ir) == custom_fixture

malformed_skills_fixture = r"""
\documentclass{article}
\begin{document}
\section{Skills}
\begin{tabularx}{\textwidth}{X X}
Broken row without a delimiter \\\
\end{tabularx}
\end{document}
"""
malformed_skills_ir = parse_resume(malformed_skills_fixture, Path("malformed_skills.tex"))
assert malformed_skills_ir.sections[0].kind == SectionKind.CUSTOM
assert malformed_skills_ir.sections[0].warnings

reordered_fixture = r"""
\documentclass{article}
\begin{document}
\section{Skills}
\begin{tabularx}{\textwidth}{X X}
Languages & Python, SQL \\\
\end{tabularx}
\section{Highlights}
\begin{itemize}
\item Moved section order should still parse correctly.
\end{itemize}
\end{document}
"""
reordered_ir = parse_resume(reordered_fixture, Path("reordered_fixture.tex"))
assert [section.title for section in reordered_ir.sections] == ["Skills", "Highlights"]
assert [section.kind for section in reordered_ir.sections] == [SectionKind.SKILLS, SectionKind.HIGHLIGHTS]
reordered_ir.sections = [reordered_ir.sections[1], reordered_ir.sections[0]]
for order, section in enumerate(reordered_ir.sections):
    section.order = order
reordered_render = render_resume(reordered_ir)
assert reordered_render.count(r"\section{Highlights}") == 1
assert reordered_render.index(r"\section{Highlights}") < reordered_render.index(r"\section{Skills}")
assert "Languages & Python, SQL" in reordered_render

unknown_section_fixture = r"""
\documentclass{article}
\begin{document}
\section{Community Work}
Did some things.
\end{document}
"""
unknown_section_ir = parse_resume(unknown_section_fixture, Path("unknown_section.tex"))
assert unknown_section_ir.sections[0].kind == SectionKind.CUSTOM
assert render_resume(unknown_section_ir) == unknown_section_fixture

incomplete_entry_fixture = r"""
\documentclass{article}
\begin{document}
\section{Experience}
\customcventry{2025}{Only one arg
\end{document}
"""
incomplete_entry_ir = parse_resume(incomplete_entry_fixture, Path("incomplete_entry.tex"))
assert incomplete_entry_ir.sections[0].kind == SectionKind.CUSTOM
assert incomplete_entry_ir.sections[0].warnings

broken_document_fixture = r"""
\documentclass{article}
\begin{document}
\section{Highlights}
\end{article}
"""
try:
    parse_resume(broken_document_fixture, Path("broken_document.tex"))
    raise AssertionError(r"Expected a document-level parsing error for a missing \end{document}.")
except ValueError as exc:
    assert r"\begin{document}" in str(exc)

assert render_text_field(TextField("R&D growth hit 10%", EscapingMode.PLAIN_TEXT)) == r"R\&D growth hit 10\%"
assert render_text_field(TextField(r"\textbf{Already LaTeX}", EscapingMode.TRUSTED_LATEX)) == r"\textbf{Already LaTeX}"

print("All inline assertions passed.")

# Partial-edit regression checks.

edited_highlights = resume_from_dict(resume_to_dict(resume_ir))
highlights_section = edited_highlights.sections[0]
highlights_section.render_mode = RenderMode.REGENERATED
highlights_section.bullets[0].text = TextField(
    r"Master's in Applied Data Science with \textbf{4+ years} of experience across analytics and ML workflows.",
    EscapingMode.TRUSTED_LATEX,
)
rendered_highlights_edit = render_resume(edited_highlights)
assert r"\textbf{4+ years}" in rendered_highlights_edit
assert resume_ir.sections[1].raw_source in rendered_highlights_edit

edited_experience = resume_from_dict(resume_to_dict(resume_ir))
experience_section = next(section for section in edited_experience.sections if section.kind == SectionKind.EXPERIENCE)
experience_section.render_mode = RenderMode.REGENERATED
experience_entry = experience_section.entries[1]
experience_entry.render_mode = RenderMode.REGENERATED
experience_entry.bullets[0].text = TextField(
    r"Improved downstream QA chatbot effectiveness by redesigning a \textbf{RAG-based legal question-answering pipeline} and lifting relevant retrieval accuracy by \textbf{2x}.",
    EscapingMode.TRUSTED_LATEX,
)
rendered_experience_edit = render_resume(edited_experience)
assert r"Justice Data and Design Lab \hfill \textit{Victoria, BC}" in rendered_experience_edit
assert "Improved downstream QA chatbot effectiveness" in rendered_experience_edit
assert experience_section.entries[0].raw_source.strip() in rendered_experience_edit

edited_projects = resume_from_dict(resume_to_dict(resume_ir))
projects_section = next(section for section in edited_projects.sections if section.kind == SectionKind.PROJECTS)
projects_section.render_mode = RenderMode.REGENERATED
project_entry = projects_section.entries[0]
project_entry.render_mode = RenderMode.REGENERATED
project_entry.title = TextField(
    r"Telecom Customer Churn Platform\enspace\faGithub",
    EscapingMode.TRUSTED_LATEX,
)
project_entry.hyperlink = "https://github.com/Koushik0901/Telecom-Customer-Churn-v2"
rendered_project_edit = render_resume(edited_projects)
assert r"\href{https://github.com/Koushik0901/Telecom-Customer-Churn-v2}{Telecom Customer Churn Platform\enspace\faGithub}" in rendered_project_edit
assert projects_section.entries[1].raw_source.strip() in rendered_project_edit

print("Partial-edit assertions passed.")


## Next steps

- If the notebook stays stable against a few additional real resume variants, extract the models, parser, renderer, and validators into a small package module.
- The next input surface for tailoring should be explicit user-provided context blocks about experience, skills, achievements, and career story, not hidden commented LaTeX.
- Formal pytest coverage can mirror the inline assertions once the notebook design has settled.
